# STUDENT PERFORMANCE ANALYSIS AND PREDICTION USING EDA AND DEEP LEARNING
## Google Colab Master Execution Notebook

**Course Code:** U21ADP05 – Exploratory Data Analysis and Visualization  
**Academic Year:** 2026–2027  
**Institution:** KPR Institute of Engineering and Technology  
**Department:** Artificial Intelligence and Data Science  
**Student Name:** Shreenidhi E.  
**Register Number:** [To be specified by student]  

---
### Notebook Capabilities:
1. **Automatic UCI Dataset Acquisition & Verification** (395 records, 33 attributes)
2. **Leakage-Free Preprocessing Pipeline** (70% Train, 15% Val, 15% Test with IQR outlier analysis)
3. **Statistical EDA & Distribution Testing** (Normality tests, ANOVA, Welch's t-tests)
4. **Publication-Quality Visualizations** (All 8 assignment charts with full academic commentary)
5. **Deep Learning MLP Model** (128-64-32 architecture, Hyperparameter tuning, EarlyStopping)
6. **Model Evaluation & Baselines** (MAE, MSE, RMSE, R² Score, actual vs predicted, residual plots)
7. **Automated Academic PDF & Word Report Generation** (with instant download button)
8. **Interactive Streamlit Web Dashboard Deployment** (directly inside Google Colab)

## Step 1: Install Required Packages in Google Colab

In [ ]:
!pip install -q reportlab python-docx streamlit pyngrok plotly seaborn
print("All required libraries successfully installed!")

## Step 2: Download the Official UCI Student Performance Dataset Automatically

In [ ]:
import os
import urllib.request
import zipfile
import io
import pandas as pd
import numpy as np

os.makedirs('dataset', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs/charts', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)
os.makedirs('report', exist_ok=True)

UCI_URL = "https://archive.ics.uci.edu/static/public/320/student+performance.zip"
dataset_path = 'dataset/student-mat.csv'

print("Fetching dataset from official UCI repository...")
req = urllib.request.Request(UCI_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=30) as resp:
    outer_zip = zipfile.ZipFile(io.BytesIO(resp.read()))
    for name in outer_zip.namelist():
        if name.endswith("student-mat.csv"):
            with open(dataset_path, "wb") as f:
                f.write(outer_zip.read(name))
            break
        elif name.endswith("student.zip"):
            inner_zip = zipfile.ZipFile(io.BytesIO(outer_zip.read(name)))
            for inner_name in inner_zip.namelist():
                if inner_name.endswith("student-mat.csv"):
                    with open(dataset_path, "wb") as f:
                        f.write(inner_zip.read(inner_name))
                    break

df = pd.read_csv(dataset_path, sep=';')
print(f"Dataset successfully loaded! Records: {df.shape[0]}, Attributes: {df.shape[1]}")
print(f"Final grade (G3) target column present: {'G3' in df.columns}")
display(df.head())

## Step 3: Complete Data Preprocessing & Leakage Prevention
- Check missing values and duplicate records
- IQR Outlier Detection: $IQR = Q_3 - Q_1$
- 70% Train, 15% Validation, 15% Test split (`random_state=42`)
- Scalers and One-Hot Encoders fitted strictly on training data

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import pickle

print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# Separate features and target G3
X = df.drop(columns=['G3'])
y = df['G3']

# 70% Train, 15% Val, 15% Test Split
X_train_raw, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Split sizes -> Train: {len(X_train_raw)}, Val: {len(X_val_raw)}, Test: {len(X_test_raw)}")

num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train_raw.select_dtypes(include=['object']).columns.tolist()

# Preprocessing Pipeline fitted strictly on training set
ct = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
])

X_train_proc = ct.fit_transform(X_train_raw)
X_val_proc = ct.transform(X_val_raw)
X_test_proc = ct.transform(X_test_raw)

print(f"Transformed feature dimension: {X_train_proc.shape[1]} features")
with open('models/preprocessing_pipeline.pkl', 'wb') as f:
    pickle.dump(ct, f)

## Step 4: Exploratory Data Analysis & Statistical Hypothesis Testing

In [ ]:
from scipy import stats

# Univariate statistics for G3
g3_series = df['G3']
print("=== G3 Univariate Statistics ===")
print(f"Mean: {g3_series.mean():.2f}")
print(f"Median: {g3_series.median():.2f}")
print(f"Std Dev: {g3_series.std():.2f}")
print(f"Skewness: {g3_series.skew():.2f}")
print(f"Kurtosis: {g3_series.kurtosis():.2f}")

# Normality Test
w, p = stats.shapiro(g3_series)
print(f"Shapiro-Wilk Test on G3: W = {w:.4f}, p = {p:.4e} -> Normality Rejected (p < 0.05)")

# One-way ANOVA: Study time vs G3
groups = [grp['G3'].values for _, grp in df.groupby('studytime')]
f_stat, p_val = stats.f_oneway(*groups)
print(f"One-Way ANOVA (Study Time -> G3): F = {f_stat:.2f}, p-value = {p_val:.4f}")

# Two-Sample t-test: Higher education desire
yes_g3 = df[df['higher'] == 'yes']['G3']
no_g3 = df[df['higher'] == 'no']['G3']
t_stat, t_p = stats.ttest_ind(yes_g3, no_g3, equal_var=False)
print(f"Welch's t-test (Higher Ed Desire): t = {t_stat:.2f}, p-value = {t_p:.4e} (Aspirants: {yes_g3.mean():.2f} vs Non: {no_g3.mean():.2f})")

## Step 5: Generate and Display All 8 Required Visualizations
Each visualization fulfills the assignment rubric with Title, Axis Labels, Observations, and Conclusions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# 1. Histogram of G3
plt.figure(figsize=(7, 4))
sns.histplot(df['G3'], bins=21, kde=True, color='#1f77b4', edgecolor='black')
plt.axvline(df['G3'].mean(), color='red', linestyle='--', label=f"Mean: {df['G3'].mean():.2f}")
plt.axvline(10, color='orange', linestyle=':', label="Pass Threshold (10)")
plt.title("1. Distribution of Final Academic Grades (G3)", fontweight='bold')
plt.xlabel("Final Grade (0 to 20)")
plt.ylabel("Student Headcount")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/1_histogram_g3.png', dpi=200)
plt.show()

# 2. Boxplot across Study Time
plt.figure(figsize=(7, 4))
labels = {1: '<2 hrs', 2: '2-5 hrs', 3: '5-10 hrs', 4: '>10 hrs'}
df_temp = df.copy()
df_temp['study_lbl'] = df_temp['studytime'].map(labels)
sns.boxplot(data=df_temp, x='study_lbl', y='G3', order=['<2 hrs', '2-5 hrs', '5-10 hrs', '>10 hrs'], palette='Blues', hue='study_lbl', legend=False)
plt.title("2. Boxplot of Final Grades Stratified by Study Time", fontweight='bold')
plt.xlabel("Weekly Study Time")
plt.ylabel("Final Grade (0-20)")
plt.tight_layout()
plt.savefig('outputs/charts/2_boxplot_g3.png', dpi=200)
plt.show()

# 3. Bar Chart Study Time
plt.figure(figsize=(7, 4))
sns.barplot(data=df_temp, x='study_lbl', y='G3', order=['<2 hrs', '2-5 hrs', '5-10 hrs', '>10 hrs'], errorbar=('ci', 95), palette='viridis', hue='study_lbl', legend=False, capsize=0.1, edgecolor='black')
plt.title("3. Average Student Performance by Weekly Study Time", fontweight='bold')
plt.xlabel("Study Time Category")
plt.ylabel("Mean Grade (G3)")
plt.tight_layout()
plt.savefig('outputs/charts/3_barchart_studytime.png', dpi=200)
plt.show()

# 4. Scatterplot Absences vs G3
plt.figure(figsize=(7, 4))
sns.regplot(data=df, x='absences', y='G3', scatter_kws={'alpha': 0.5, 'color': '#2b5c8f'}, line_kws={'color': 'crimson', 'label': 'Linear Trendline'})
plt.title("4. Relationship Between School Absences and Final Grade", fontweight='bold')
plt.xlabel("Total Absences (Days)")
plt.ylabel("Final Grade (0-20)")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/4_scatterplot_absences.png', dpi=200)
plt.show()

# 5. Correlation Heatmap
plt.figure(figsize=(9, 6))
num_vars = ['age', 'Medu', 'Fedu', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
sns.heatmap(df[num_vars].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title("5. Correlation Matrix Heatmap of Numerical Student Features", fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/5_correlation_heatmap.png', dpi=200)
plt.show()

# 6. Pairplot
pair_vars = ['age', 'studytime', 'failures', 'absences', 'G1', 'G3']
g = sns.pairplot(df[pair_vars], diag_kind='kde', plot_kws={'alpha': 0.5})
g.fig.suptitle("6. Pairplot of Selected Numerical Predictors", y=1.02, fontweight='bold')
g.savefig('outputs/charts/6_pairplot_numerical.png', dpi=200)
plt.show()

# 7. Categorical Factors Countplot
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
attrs = [('higher', 'Desire Higher Ed'), ('internet', 'Home Internet'), ('schoolsup', 'School Support'), ('paid', 'Paid Classes')]
for ax, (col, title) in zip(axes.flatten(), attrs):
    sns.countplot(data=df, x=col, hue=col, palette='Set2', legend=False, edgecolor='black', ax=ax)
    ax.set_title(title, fontweight='bold')
plt.suptitle("7. Distribution of Key Categorical Factors", y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/charts/7_countplot_categorical.png', dpi=200)
plt.show()

# 8. Comparative Density
plt.figure(figsize=(7, 4))
sns.kdeplot(data=df[df['higher'] == 'yes']['G3'], label="Wants Higher Ed (Yes)", color='#2ca02c', fill=True, alpha=0.3)
sns.kdeplot(data=df[df['higher'] == 'no']['G3'], label="Wants Higher Ed (No)", color='#d62728', fill=True, alpha=0.3)
plt.title("8. Comparative Density of Final Grades by Higher Education Aspiration", fontweight='bold')
plt.xlabel("Final Grade (G3: 0-20)")
plt.ylabel("Kernel Density")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/8_distribution_comparison.png', dpi=200)
plt.show()

print("All 8 required visualizations generated and saved!")

## Step 6: Deep Learning Model (Multi-Layer Perceptron) Training
Architecture:
- Input: Processed feature dimension
- Hidden Layer 1: 128 neurons, ReLU, Dropout 0.2
- Hidden Layer 2: 64 neurons, ReLU, Dropout 0.2
- Hidden Layer 3: 32 neurons, ReLU
- Output: 1 neuron, Linear
- Optimizer: Adam, Loss: MSE, Metric: MAE, EarlyStopping & ModelCheckpoint

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential([
    layers.Input(shape=(X_train_proc.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='linear')
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])
model.summary()

cb_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
cb_ckpt = callbacks.ModelCheckpoint('models/best_model.keras', monitor='val_loss', save_best_only=True)

history = model.fit(
    X_train_proc, y_train,
    validation_data=(X_val_proc, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[cb_stop, cb_ckpt],
    verbose=1
)
print("MLP training completed and optimal weights saved!")

## Step 7: Model Evaluation, Baselines & Evaluation Visualizations

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Evaluate Deep Learning MLP
test_preds_mlp = model.predict(X_test_proc).flatten()
mae_mlp = mean_absolute_error(y_test, test_preds_mlp)
mse_mlp = mean_squared_error(y_test, test_preds_mlp)
rmse_mlp = np.sqrt(mse_mlp)
r2_mlp = r2_score(y_test, test_preds_mlp)

# Evaluate Baselines
lr = LinearRegression().fit(X_train_proc, y_train)
lr_preds = lr.predict(X_test_proc)
mae_lr = mean_absolute_error(y_test, lr_preds)
rmse_lr = np.sqrt(mean_squared_error(y_test, lr_preds))
r2_lr = r2_score(y_test, lr_preds)

rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train_proc, y_train)
rf_preds = rf.predict(X_test_proc)
mae_rf = mean_absolute_error(y_test, rf_preds)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_preds))
r2_rf = r2_score(y_test, rf_preds)

comparison_table = pd.DataFrame([
    {"Model": "Linear Regression (Baseline)", "MAE": mae_lr, "RMSE": rmse_lr, "R² Score": r2_lr},
    {"Model": "Random Forest Regressor", "MAE": mae_rf, "RMSE": rmse_rf, "R² Score": r2_rf},
    {"Model": "Deep Learning MLP (Best Model)", "MAE": mae_mlp, "RMSE": rmse_mlp, "R² Score": r2_mlp}
])
display(comparison_table.round(4))

# 4 Evaluation Charts
# Chart 1: Loss curves
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='Train Loss (MSE)', color='#1f77b4', linewidth=2)
plt.plot(history.history['val_loss'], label='Val Loss (MSE)', color='#ff7f0e', linestyle='--', linewidth=2)
plt.title("Figure 9: Training Loss vs Validation Loss Across Epochs", fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/eval_loss_curves.png', dpi=200)
plt.show()

# Chart 2: Actual vs Predicted
plt.figure(figsize=(7, 4.5))
plt.scatter(y_test, test_preds_mlp, color='#2b5c8f', alpha=0.7, edgecolors='black')
plt.plot([0, 20], [0, 20], color='red', linestyle='--', label='Ideal Fit (y=x)')
plt.title(f"Figure 10: Actual vs Predicted Grades (R² = {r2_mlp:.3f})", fontweight='bold')
plt.xlabel("Actual Final Grade (0-20)")
plt.ylabel("Predicted Final Grade (0-20)")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/eval_actual_vs_predicted.png', dpi=200)
plt.show()

# Chart 3: Residuals Distribution
plt.figure(figsize=(7, 4))
residuals = y_test - test_preds_mlp
sns.histplot(residuals, bins=15, kde=True, color='#4c72b0', edgecolor='black')
plt.axvline(0, color='red', linestyle='--', label='Zero Error Line')
plt.title(f"Figure 11: Prediction Error Distribution (Residual Mean: {residuals.mean():.2f})", fontweight='bold')
plt.xlabel("Residual Error (Actual - Predicted)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/eval_error_distribution.png', dpi=200)
plt.show()

# Chart 4: Model Comparison
plt.figure(figsize=(7, 4))
x_idx = np.arange(3)
w = 0.35
plt.bar(x_idx - w/2, [mae_lr, mae_rf, mae_mlp], w, label='MAE', color='#3b6998', edgecolor='black')
plt.bar(x_idx + w/2, [rmse_lr, rmse_rf, rmse_mlp], w, label='RMSE', color='#e8743b', edgecolor='black')
plt.xticks(x_idx, ['Linear Reg', 'Random Forest', 'Deep Learning MLP'])
plt.title("Figure 12: Benchmark Comparison of Error Metrics", fontweight='bold')
plt.ylabel("Metric Value")
plt.legend()
plt.tight_layout()
plt.savefig('outputs/charts/eval_baseline_comparison.png', dpi=200)
plt.show()

## Step 8: Generate Academic PDF and Word Reports with Direct Colab Download

In [ ]:
# Execute python report generator
!python -c "import sys; sys.path.append('src'); from report_generator import generate_all_reports; generate_all_reports()"

print("Project reports generated successfully:")
print(" - report/project_report.pdf")
print(" - report/project_report.docx")

# In Google Colab, you can instantly download both files:
try:
    from google.colab import files
    print("Downloading PDF report to your computer...")
    files.download('report/project_report.pdf')
    print("Downloading Word report (.docx) to your computer...")
    files.download('report/project_report.docx')
except Exception as e:
    print("Running outside Colab or automatic download skipped:", e)

## Step 9: Launch the Streamlit Web Dashboard inside Google Colab
Run the cell below to host the interactive Streamlit dashboard directly from Google Colab using `localtunnel`.

In [ ]:
# Get public IP for LocalTunnel authentication password
!curl -s ipv4.icanhazip.com

# Start Streamlit in background and expose via localtunnel
import subprocess
import time

print("Starting Streamlit dashboard in background on port 8501...")
streamlit_process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
time.sleep(3)

print("\n--- HOW TO ACCESS STREAMLIT FROM GOOGLE COLAB ---")
print("1. Click the localtunnel URL generated below.")
print("2. Paste the IP address displayed above into the 'Tunnel Password' prompt.")
print("3. The complete STUDENT PERFORMANCE ANALYTICS application will open!\n")

!npx localtunnel --port 8501